# 🧩 Лабораторная работа 6. CNN на MNIST

Цель: понять свёрточные слои на практике и обучить CNN на том же MNIST, который использовался в главе 5.


# 1. Импорт

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

RANDOM_SEED = 42
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001

torch.manual_seed(RANDOM_SEED)
print("PyTorch:", torch.__version__)

# 2. MNIST и DataLoader

In [ ]:
DATA_DIR = Path("../../Datasets/mnist")

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

# 3. Один Batch

In [ ]:
images, targets = next(iter(train_loader))

print("Images:", images.shape)
print("Targets:", targets.shape)

# 4. Один Conv2d

In [ ]:
conv = nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3,
    padding=1,
)

conv_output = conv(images)

print("Input:", images.shape)
print("After Conv:", conv_output.shape)

# 5. ReLU и Pooling

In [ ]:
relu = nn.ReLU()
pool = nn.MaxPool2d(2)

activated = relu(conv_output)
pooled = pool(activated)

print("After ReLU:", activated.shape)
print("After Pool:", pooled.shape)

# 6. Исходная цифра

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(images[0].squeeze(), cmap="gray")
plt.title(f"Target: {targets[0].item()}")
plt.axis("off")
plt.show()

# 7. Feature Maps до обучения

In [ ]:
maps = conv_output[0].detach().cpu()

plt.figure(figsize=(12, 6))

for i in range(min(8, maps.shape[0])):
    plt.subplot(2, 4, i + 1)
    plt.imshow(maps[i], cmap="gray")
    plt.title(f"Map {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 8. Создаём CNN

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, images):
        features = self.features(images)
        return self.classifier(features)


model = MNISTCNN()
print(model)

# 9. Количество параметров

In [ ]:
total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("Всего параметров:", total_parameters)

# 10. Shapes слой за слоем

In [ ]:
x = images[:1]

print("Input:", x.shape)

for i, layer in enumerate(model.features):
    x = layer(x)
    print(i, layer.__class__.__name__, "→", x.shape)

for i, layer in enumerate(model.classifier):
    x = layer(x)
    print("classifier", i, layer.__class__.__name__, "→", x.shape)

# 11. Функции обучения и оценки

In [ ]:
def train_one_epoch(model, loader, loss_function, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = loss_function(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = logits.argmax(dim=1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            logits = model(images)
            loss = loss_function(logits, targets)

            total_loss += loss.item()
            predictions = logits.argmax(dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)

    return total_loss / len(loader), correct / total

# 12. Device, Loss и Optimizer

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Device:", device)

# 13. Обучаем CNN

In [ ]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
}

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        loss_function,
        optimizer,
        device,
    )

    test_loss, test_accuracy = evaluate(
        model,
        test_loader,
        loss_function,
        device,
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_accuracy:.4f}"
    )

# 14. Loss

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history["train_loss"], label="Train")
plt.plot(history["test_loss"], label="Test")
plt.title("CNN MNIST Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# 15. Accuracy

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history["train_accuracy"], label="Train")
plt.plot(history["test_accuracy"], label="Test")
plt.title("CNN MNIST Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 16. Предсказания

In [ ]:
model.eval()

images, targets = next(iter(test_loader))
device_images = images.to(device)

with torch.no_grad():
    logits = model(device_images)
    predictions = logits.argmax(dim=1).cpu()

plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"T={targets[i].item()} / P={predictions[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 17. Feature Maps обученной CNN

In [ ]:
single_image = images[:1].to(device)

with torch.no_grad():
    first_conv = model.features[0](single_image)
    first_relu = model.features[1](first_conv)

maps = first_relu[0].cpu()

plt.figure(figsize=(12, 6))

for i in range(min(8, maps.shape[0])):
    plt.subplot(2, 4, i + 1)
    plt.imshow(maps[i], cmap="gray")
    plt.title(f"Feature {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 18. Сохраняем модель

In [ ]:
MODEL_PATH = Path("mnist_cnn.pth")

torch.save(
    model.state_dict(),
    MODEL_PATH,
)

print("Сохранено:", MODEL_PATH.resolve())

# 19. 📌 Что нужно запомнить

```text
Image
↓
Conv2d
↓
ReLU
↓
Pooling
↓
Conv2d
↓
ReLU
↓
Pooling
↓
Flatten
↓
Linear
↓
Logits
```


# 20. 🧩 Эксперименты

Попробуй:
- заменить `8 → 16` на `16 → 32`;
- изменить `kernel_size`;
- убрать один Pooling;
- сравнить CNN с MLP из главы 5;
- посмотреть Feature Maps для разных цифр.


# 21. ➡️ Следующая глава

# Глава 7. Текст, токены и Embeddings
